# Regrid WRFOUT files 
This code regrids the WRFOUT files to match the reanalysis data so that they work with QTRACK. 6hrly 1x1 degree grid spacing. 

In [1]:
import pkg_resources
from wrf import (to_np, interplevel, geo_bounds, getvar, smooth2d, get_cartopy, cartopy_xlim,
                 cartopy_ylim, latlon_coords, destagger)
import wrf
import numpy as np
import xarray as xr
import pandas as pd
from datetime import date
from numpy import absolute, exp, log
from netCDF4 import Dataset, num2date, date2num

# Any import of metpy will activate the accessors
from metpy.units import units
import os
import glob
import xesmf as xe

/glade/derecho/scratch/athornton/tmp/ipykernel_18798/2911559653.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [16]:
## is this a restart run?
restart = False
# what is the base initialization time? (ens member)
init_time = pd.to_datetime('2020-09-03 12:00:00')
init_string = init_time.strftime('%d%H')
year = init_time.strftime('%Y')
# what time did you turn on the fluxes?
# ignore this if it is not a restart run
fluxon_time = pd.to_datetime('2020-09-05 12:00:00')
# this string is used to find that experiment
string = fluxon_time.strftime('%d%H')
# if this is not a restart run adjust the following
flux = 'fluxoff' # or 'fluxon' 

In [17]:
if year == '2011':
    subdir = 'long_lived_case'
    if int(init_string) % 2 == 0:
        end_time = pd.to_datetime('2011-08-29 12:00') # Even 
    else:
        end_time = pd.to_datetime('2011-08-29 09:00') # Odd 
else:
    subdir = 'cent_atl_case'
    end_time = pd.to_datetime('2020-09-09 12:00')  
    
# for indexing...
date_list = pd.date_range(start=init_time, end=end_time, freq='3h')
fluxon_index = np.where(date_list==fluxon_time)[0][0]
response = fluxon_index + 3 # 3 days later, to respond to fluxes on

# Set directory where wrfout files reside, and list the files for processing.  Set up for a directory with only wrfout files.
if restart == True:
    plt_name = 'rst_on'+string+'z'
    f = pd.Timedelta(init_time - fluxon_time).total_seconds() 
    hours = int((f / 3600)*-1)
    run_name = 'rst_on'+str(hours)
else:
    run_name = flux
    plt_name = run_name
    hours = run_name

os.chdir("/glade/campaign/univ/uncs0067/flux_experiments/"+subdir+"/init"+init_string+"z/"+run_name+"/")
plotsdir = '/glade/u/home/athornton/wrf_visualization/plots/restart/init'+init_string+'z/'+plt_name+'/'

title = init_string+", "+run_name+", "+string+"z"
save_name = run_name +"_"+ init_string


In [18]:
save_name

'fluxoff_0312'

In [19]:
datafiles = (sorted(glob.glob("wrfout_d01_*")))
numfiles=len(datafiles)

In [20]:
numfiles

49

In [21]:
times = []
u_list = []
v_list = []
for i in range(0,numfiles):
    wrf_out_data = xr.open_dataset(datafiles[i])  
    ncfile = Dataset(datafiles[i])
    
    #First, lets find the data data assocaited with all of them
    time = wrf_out_data['XTIME'][0].values
    datetime = time.astype(np.datetime64)
    ts = pd.to_datetime(str(datetime))
    d = ts.strftime('%Y-%m-%d %H:%M')
    Time=wrf.extract_times(ncfile, timeidx=0, method='cat', squeeze=True, cache=None, meta=False, do_xtime=False)
    timestr=(str(Time))
    # Set up one time string for plot titles, another for file names
    titletime=(timestr[0:10]+' '+timestr[11:16])
    filetime=(timestr[0:10]+'_'+timestr[11:13])
    times.append(timestr)
    print(filetime)

    # Get variables
    ua = wrf_out_data["U"][0]
    va = wrf_out_data["V"][0]
    p = wrf_out_data["P"][0]
    
    # Unstagger the winds to match the pressure grid
    ua = destagger(ua, stagger_dim=2)  # Unstagger U in the x-direction
    va = destagger(va, stagger_dim=1)  # Unstagger V in the x-direction
    
    u_winds = xr.DataArray(ua)
    v_winds = xr.DataArray(va)
    
    u_winds = u_winds.sel(dim_0=17) # select 699 hPa winds
    v_winds = v_winds.sel(dim_0=17) # select 699 hPa winds

    u_list.append(u_winds)
    v_list.append(v_winds)
    

2020-09-03_12
2020-09-03_15
2020-09-03_18
2020-09-03_21
2020-09-04_00
2020-09-04_03
2020-09-04_06
2020-09-04_09
2020-09-04_12
2020-09-04_15
2020-09-04_18
2020-09-04_21
2020-09-05_00
2020-09-05_03
2020-09-05_06
2020-09-05_09
2020-09-05_12
2020-09-05_15
2020-09-05_18
2020-09-05_21
2020-09-06_00
2020-09-06_03
2020-09-06_06
2020-09-06_09
2020-09-06_12
2020-09-06_15
2020-09-06_18
2020-09-06_21
2020-09-07_00
2020-09-07_03
2020-09-07_06
2020-09-07_09
2020-09-07_12
2020-09-07_15
2020-09-07_18
2020-09-07_21
2020-09-08_00
2020-09-08_03
2020-09-08_06
2020-09-08_09
2020-09-08_12
2020-09-08_15
2020-09-08_18
2020-09-08_21
2020-09-09_00
2020-09-09_03
2020-09-09_06
2020-09-09_09
2020-09-09_12


In [22]:
times = pd.to_datetime(times)
times

DatetimeIndex(['2020-09-03 12:00:00', '2020-09-03 15:00:00',
               '2020-09-03 18:00:00', '2020-09-03 21:00:00',
               '2020-09-04 00:00:00', '2020-09-04 03:00:00',
               '2020-09-04 06:00:00', '2020-09-04 09:00:00',
               '2020-09-04 12:00:00', '2020-09-04 15:00:00',
               '2020-09-04 18:00:00', '2020-09-04 21:00:00',
               '2020-09-05 00:00:00', '2020-09-05 03:00:00',
               '2020-09-05 06:00:00', '2020-09-05 09:00:00',
               '2020-09-05 12:00:00', '2020-09-05 15:00:00',
               '2020-09-05 18:00:00', '2020-09-05 21:00:00',
               '2020-09-06 00:00:00', '2020-09-06 03:00:00',
               '2020-09-06 06:00:00', '2020-09-06 09:00:00',
               '2020-09-06 12:00:00', '2020-09-06 15:00:00',
               '2020-09-06 18:00:00', '2020-09-06 21:00:00',
               '2020-09-07 00:00:00', '2020-09-07 03:00:00',
               '2020-09-07 06:00:00', '2020-09-07 09:00:00',
               '2020-09-

In [23]:
u_wind = np.array(u_list)
v_wind = np.array(v_list)

In [24]:
lats, lons = latlon_coords(wrf_out_data['P'][0])

In [25]:
# Slice these arrays so they work with the new dataset
lat = to_np(lats)[:,0]
lon = to_np(lons)[0]

In [26]:
# Create one dataset with u, v winds at each lat, lon, time coordinate for this ensemble member
ds = xr.Dataset( 
    data_vars=dict(
        u=(["time","lat","lon"], u_wind.data),
        v=(["time","lat","lon"], v_wind.data),
    ),
    coords=dict(
        time=("time", times),
        lon=("lon", lon),
        lat=("lat", lat),
    ),
)

In [27]:
# Regrid the lats and lons
ds_out = xr.Dataset( 
    {
        "latitude": (["latitude"], np.arange(-1,37, 1.0),  {"units": "degrees_north"}),
        "longitude": (["longitude"], np.arange(-120, 20, 1.0), {"units": "degrees_east"}),

    }
)

# Grab the lats and lons from the old data
ds_in = xr.Dataset(
    {
        "latitude": (["latitude"], ds.lat.values,  {"units": "degrees_north"}),
        "longitude": (["longitude"], ds.lon.values, {"units": "degrees_east"}),
    }
)
regridder = xe.Regridder(ds_in, ds_out, "conservative")

In [29]:
# apply regridding and save file
dat_out = regridder(ds, keep_attrs=True)
path_out = "/glade/u/home/athornton/qtrack/wind_files/"
file_out = path_out +'wrfout_regrid_'+save_name+'.nc'
dat_out.to_netcdf(path=file_out, format='NETCDF4', mode='w')

/glade/u/apps/opt/conda/envs/npl-2025b/lib/python3.12/site-packages/xesmf/frontend.py:716: UserWarning: Using dimensions ('lat', 'lon') from data variable u as the horizontal dimensions for the regridding.
  warnings.warn(


In [30]:
file_out

'/glade/u/home/athornton/qtrack/wind_files/wrfout_regrid_fluxoff_0312.nc'